# Train segmentation model

## Training the segmentation head from embeddings

In this notebook you can train the segmentation model using pre-computed embeddings via LightningCLI using configurations in `configs/train_embedding_classifier.yaml`. Modify the batch size, learning rate, and other hyperparameters in the configuration file as needed.

This notebook uses a CSV logger by default for training, validation and test results. A [WandB logger](https://lightning.ai/docs/pytorch/stable/extensions/generated/lightning.pytorch.loggers.WandbLogger.html#lightning.pytorch.loggers.WandbLogger) can also be used. If you prefer the latter, please switch the loggers and update the entity of the WandB logger configuration in `configs/train_embedding_classifier.yaml`.


In [1]:
import os
from pathlib import Path

def find_project_root(marker='claymodel'):
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / marker).exists():
            return path
    raise FileNotFoundError(f"Project root not found")

os.chdir(find_project_root())


In [ ]:
from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.cli import LightningArgumentParser, instantiate_class

from claymodel.finetune.flood_detection.embedding_gfm_classifier import EmbeddingClassifierGFM
from claymodel.finetune.flood_detection import EmbeddingDataModuleGFM

from datetime import datetime

seed_everything(42)  # your seed here

def Train_segmentation_from_embeddings(config_path, test_after_training=False):
    
    objects = ["callbacks", "logger", "plugins"]

    # Create argument parser similar to LightningCLI
    parser = LightningArgumentParser()
    parser.add_lightning_class_args(EmbeddingClassifierGFM, "model")
    parser.add_lightning_class_args(EmbeddingDataModuleGFM, "data")
    parser.add_lightning_class_args(Trainer, "trainer")
    
    # Parse the config file
    config = parser.parse_path(config_path)

    trainer_config = dict(config["trainer"])
    
    # Instantiate objects
    for object in objects:
        if object in trainer_config and trainer_config[object]:
            callbacks = []
            for callback_config in trainer_config[object]:
                if hasattr(callback_config, 'class_path') and hasattr(callback_config, 'init_args'):
                    # This is a Namespace object from CLI parsing
                    if object == "logger" and callback_config.class_path == "lightning.pytorch.loggers.CSVLogger":
                        callback_config.init_args['version'] = datetime.now().strftime("%Y%m%d_%H%M%S")
                    if object == "callbacks" and callback_config.class_path == "lightning.pytorch.callbacks.ModelCheckpoint":
                        callback_config.init_args['dirpath'] = os.path.join(callback_config.init_args['dirpath'], datetime.now().strftime("%Y%m%d_%H%M%S"))
                    callback = instantiate_class((), callback_config)
                    callbacks.append(callback)                    
                elif isinstance(callback_config, dict) and "class_path" in callback_config:
                    # This is a dictionary configuration
                    callback = instantiate_class((), callback_config)
                    callbacks.append(callback)
                else:
                    # Already an instantiated callback
                    callbacks.append(callback_config)
            
            trainer_config[object] = callbacks

    embedd_dirs = ["train_embedd_dir", "val_embedd_dir", "test_embedd_dir"]
    for ed in embedd_dirs:
        if ed in config["data"] and config["data"][ed][-1] == '/':
            config["data"][ed] = config["data"][ed][:-1]
        config["data"][ed] += "_" + config["model"]["ckpt_path"].replace('/', '_').split('.')[0]
    
    # Create instances
    model = EmbeddingClassifier(**config["model"])
    datamodule = EmbeddingDataModule(**config["data"])
    trainer = Trainer(**trainer_config)

    result = trainer.fit(model, datamodule)
    if test_after_training:
        result = trainer.test(model, datamodule)
        print(model.confusion_matrix)

    print("Results:", result)
    
    return model, datamodule, trainer, result


Seed set to 42


## Training the model

Run the training process. The model will be trained using the configuration specified in `configs/train_embedding_classifier.yaml`. Set `test_after_training=True` to also run evaluation on the test set after training.


In [3]:
CONFIG_PATH = "configs/train_embedding_classifier.yaml"
model, *_ = Train_segmentation_from_embeddings(CONFIG_PATH)


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/ignacio/Code/Clay-foundation/model/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:231: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name              | Type                      | Params | Mode 
------------------------------------------------------------------------
0 | model             | EmbeddingSegmentationHead | 26.0 M | train
1 | loss_fn           | FocalLoss                 | 0      | train
2 | iou               | MulticlassJaccardIndex    | 0      | train
3 | f1                | MulticlassF1Score         | 0      | train
4 | _confusion_matrix | MulticlassConfusionMatrix | 0      | train
------------------------------------------------------------------------
26.0 M    Trainable 

🏗️  EmbeddingClassifier initialized:
   Input: 1024 channels, 8 spatial
   Output: 6 classes, (224, 224) spatial


/home/ignacio/Code/Clay-foundation/model/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Training: |          | 0/? [00:00<?, ?it/s]

/home/ignacio/Code/Clay-foundation/model/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 32. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/home/ignacio/Code/Clay-foundation/model/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 8. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 0, global step 32: 'val/iou' reached 0.83530 (best 0.83530), saving model to '/home/ignacio/Code/Clay-foundation/model/checkpoints/segment_embeddings/20250923_070441/embedding-segment_epoch-00_val-iou-0.8353.ckpt' as top 2
`Trainer.fit` stopped: `max_epochs=1` reached.


Results: None
